# 01 - Dataset exploration

This notebook *reads saved artifacts*; it does not recompute anything. Run the
pipeline first:

```bash
python -m eeg_research_agent.cli run --mode smoke
```

Everything here is provenance-aware: the corpus is tagged with `data_origin`,
and an `synthetic` origin means the numbers describe the surrogate generator,
not human sleep.

In [ ]:
import json
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
summary = json.loads((ROOT / 'artifacts/tables/dataset_summary.json').read_text())
print('dataset        :', summary['dataset'])
print('data origin    :', summary['data_origin'], '  <-- synthetic = surrogate, not human EEG')
print('derivation     :', summary['selected_channel'])
print('sampling rate  :', summary['sampling_frequency'], 'Hz')
print('epoch duration :', summary['epoch_duration'], 's')
print('subjects/epochs:', summary['subject_count'], '/', summary['epoch_count'])
print('stages present :', summary['available_stages'])
print('stages missing :', summary['missing_stages'])

## Class distribution

Sleep data is heavily imbalanced, which is why every later metric is reported
as *balanced* accuracy or macro F1 rather than plain accuracy.

In [ ]:
counts = pd.Series(summary['class_distribution']).sort_values(ascending=False)
frame = counts.to_frame('epochs')
frame['fraction'] = (frame['epochs'] / frame['epochs'].sum()).round(4)
display(frame)
counts.plot(kind='bar', title='Epochs per sleep stage', ylabel='epochs')

## Recording quality and annotation mapping

`quality_report` is computed on the preprocessed signal. Annotations that did not
map to a canonical stage are counted as *unmapped* (never coerced into a stage);
the per-recording detail lives in the acquisition manifest.

In [ ]:
print(json.dumps(summary['quality_report'], indent=2))
manifest = json.loads((ROOT / 'artifacts/execution_logs/execution_manifest.json').read_text())
print('stages run :', manifest.get('stages'))
print('metrics    :', json.dumps(manifest.get('metrics'), indent=2))

## Split integrity

Splits are by **subject**. The verification stage fails the run if a subject
appears under two split labels, so this is a checkable property, not a promise.

In [ ]:
splits = json.loads((ROOT / 'data/metadata/subject_splits.json').read_text())
for name in ('train', 'validation', 'test'):
    print(f"{name:<11}", splits[f'{name}_subjects'])
print('seed       ', splits['seed'])